# SQL入门 —— SELECT/WHERE/ORDER BY/LIMIT（练习）

In [1]:
import duckdb

%load_ext sql
%sql duckdb://

Connecting to 'duckdb://'

In [5]:
# 1. 最基础的SELECT
# 看前5行 —— 任何新数据集都先这么干
duckdb.sql("SELECT * FROM '../data/sales.csv' LIMIT 5")

┌──────────┬─────────────┬──────────┬───────────┬──────────┬───────┬────────────┬─────────┬───────┐
│ order_id │ customer_id │ product  │ category  │ quantity │ price │ order_date │ country │ total │
│ varchar  │   varchar   │ varchar  │  varchar  │  int64   │ int64 │    date    │ varchar │ int64 │
├──────────┼─────────────┼──────────┼───────────┼──────────┼───────┼────────────┼─────────┼───────┤
│ O1000    │ C007        │ Laptop   │ Computer  │        5 │   599 │ 2024-01-01 │ US      │  2995 │
│ O1001    │ C004        │ Mouse    │ Accessory │        3 │   999 │ 2024-01-01 │ UK      │  2997 │
│ O1002    │ C005        │ Keyboard │ Accessory │        4 │   299 │ 2024-01-01 │ China   │  1196 │
│ O1003    │ C007        │ Mouse    │ Accessory │        3 │    99 │ 2024-01-01 │ Germany │   297 │
│ O1004    │ C003        │ Monitor  │ Computer  │        1 │  1999 │ 2024-01-02 │ Germany │  1999 │
└──────────┴─────────────┴──────────┴───────────┴──────────┴───────┴────────────┴─────────┴───────┘

In [6]:
# *是“所有列”，但实际工作中尽量列出具体列（性能 + 可读性）
duckdb.sql("""
    SELECT order_id, customer_id, product, total
    FROM '../data/sales.csv'
    LIMIT 5
""")

┌──────────┬─────────────┬──────────┬───────┐
│ order_id │ customer_id │ product  │ total │
│ varchar  │   varchar   │ varchar  │ int64 │
├──────────┼─────────────┼──────────┼───────┤
│ O1000    │ C007        │ Laptop   │  2995 │
│ O1001    │ C004        │ Mouse    │  2997 │
│ O1002    │ C005        │ Keyboard │  1196 │
│ O1003    │ C007        │ Mouse    │   297 │
│ O1004    │ C003        │ Monitor  │  1999 │
└──────────┴─────────────┴──────────┴───────┘

In [ ]:
# 2. 列的重命名（AS）和运算
# 用AS给列起别名
duckdb.sql("""
    SELECT
        order_id,
        total AS revenue
    FROM '../data/sales.csv'
    LIMIT 5
""")

┌──────────┬─────────┐
│ order_id │ revenue │
│ varchar  │  int64  │
├──────────┼─────────┤
│ O1000    │    2995 │
│ O1001    │    2997 │
│ O1002    │    1196 │
│ O1003    │     297 │
│ O1004    │    1999 │
└──────────┴─────────┘

In [8]:
# 直接在SELECT里计算 —— SQL的列可以是表达式
duckdb.sql("""
    SELECT
        order_id,
        quantity,
        price,
        quantity * price AS total_check,
        total
    FROM '../data/sales.csv'
    LIMIT 5
""")

┌──────────┬──────────┬───────┬─────────────┬───────┐
│ order_id │ quantity │ price │ total_check │ total │
│ varchar  │  int64   │ int64 │    int64    │ int64 │
├──────────┼──────────┼───────┼─────────────┼───────┤
│ O1000    │        5 │   599 │        2995 │  2995 │
│ O1001    │        3 │   999 │        2997 │  2997 │
│ O1002    │        4 │   299 │        1196 │  1196 │
│ O1003    │        3 │    99 │         297 │   297 │
│ O1004    │        1 │  1999 │        1999 │  1999 │
└──────────┴──────────┴───────┴─────────────┴───────┘

In [ ]:
# 3. WHERE过滤
# 单条件过滤
duckdb.sql("""
    SELECT order_id, country, total
    FROM '../data/sales.csv'
    WHERE country = 'UK'
    LIMIT 10
""")

# 关键：SQL里字符串用单引号'...'，双引号是给列名/表名用的（别弄混）

┌──────────┬─────────┬───────┐
│ order_id │ country │ total │
│ varchar  │ varchar │ int64 │
├──────────┼─────────┼───────┤
│ O1001    │ UK      │  2997 │
│ O1007    │ UK      │   396 │
│ O1008    │ UK      │  6495 │
│ O1017    │ UK      │   297 │
│ O1019    │ UK      │  1299 │
│ O1022    │ UK      │   495 │
│ O1025    │ UK      │  2598 │
│ O1026    │ UK      │    99 │
│ O1036    │ UK      │  1299 │
│ O1037    │ UK      │   599 │
└──────────┴─────────┴───────┘
  10 rows          3 columns

In [10]:
# 数值比较
duckdb.sql("""
    SELECT order_id, total
    FROM '../data/sales.csv'
    WHERE total > 3000
""")

┌──────────┬───────┐
│ order_id │ total │
│ varchar  │ int64 │
├──────────┼───────┤
│ O1006    │  3996 │
│ O1008    │  6495 │
│ O1016    │  4995 │
│ O1018    │  7996 │
│ O1031    │  3998 │
│ O1040    │  4995 │
│ O1046    │  6495 │
│ O1047    │  5196 │
│ O1050    │  7996 │
│ O1052    │  5196 │
│   ·      │    ·  │
│   ·      │    ·  │
│   ·      │    ·  │
│ O1164    │  3996 │
│ O1165    │  6495 │
│ O1167    │  3996 │
│ O1169    │  9995 │
│ O1175    │  6495 │
│ O1179    │  3998 │
│ O1182    │  7996 │
│ O1189    │  7996 │
│ O1192    │  6495 │
│ O1195    │  3998 │
└──────────┴───────┘
      55 rows     
     (20 shown)    

In [11]:
# 多条件：AND / OR
duckdb.sql("""
    SELECT order_id, country, total
    FROM '../data/sales.csv'
    WHERE country = 'UK' AND total > 2000
""")

┌──────────┬─────────┬───────┐
│ order_id │ country │ total │
│ varchar  │ varchar │ int64 │
├──────────┼─────────┼───────┤
│ O1001    │ UK      │  2997 │
│ O1008    │ UK      │  6495 │
│ O1025    │ UK      │  2598 │
│ O1042    │ UK      │  2598 │
│ O1043    │ UK      │  2997 │
│ O1044    │ UK      │  2598 │
│ O1046    │ UK      │  6495 │
│ O1052    │ UK      │  5196 │
│ O1056    │ UK      │  2396 │
│ O1066    │ UK      │  7996 │
│   ·      │ ·       │    ·  │
│   ·      │ ·       │    ·  │
│   ·      │ ·       │    ·  │
│ O1145    │ UK      │  7996 │
│ O1146    │ UK      │  5997 │
│ O1147    │ UK      │  2396 │
│ O1150    │ UK      │  2997 │
│ O1157    │ UK      │  4995 │
│ O1167    │ UK      │  3996 │
│ O1169    │ UK      │  9995 │
│ O1178    │ UK      │  2598 │
│ O1182    │ UK      │  7996 │
│ O1184    │ UK      │  2995 │
└──────────┴─────────┴───────┘
  37 rows          3 columns
  (20 shown)                 

In [12]:
# IN：相当于Python的in[...]
duckdb.sql("""
    SELECT order_id, country
    FROM '../data/sales.csv'
    WHERE country in ('UK', 'US', 'China')
    LIMIT 10
""")

┌──────────┬─────────┐
│ order_id │ country │
│ varchar  │ varchar │
├──────────┼─────────┤
│ O1000    │ US      │
│ O1001    │ UK      │
│ O1002    │ China   │
│ O1005    │ US      │
│ O1006    │ US      │
│ O1007    │ UK      │
│ O1008    │ UK      │
│ O1011    │ China   │
│ O1012    │ China   │
│ O1013    │ US      │
└──────────┴─────────┘
  10 rows  2 columns

In [13]:
# BETWEEN：范围（包含两端）
duckdb.sql("""
    SELECT order_id, total
    FROM '../data/sales.csv'
    WHERE total BETWEEN 1000 AND 2000
    LIMIT 10
""")

┌──────────┬───────┐
│ order_id │ total │
│ varchar  │ int64 │
├──────────┼───────┤
│ O1002    │  1196 │
│ O1004    │  1999 │
│ O1005    │  1299 │
│ O1010    │  1797 │
│ O1012    │  1999 │
│ O1015    │  1999 │
│ O1019    │  1299 │
│ O1027    │  1198 │
│ O1028    │  1998 │
│ O1030    │  1998 │
└──────────┴───────┘
      10 rows     

In [14]:
# LIKE：字符串模糊匹配（%表示任意字符，_表示一个字符）
duckdb.sql("""
    SELECT product
    FROM '../data/sales.csv'
    WHERE product LIKE 'M%' -- 以M开头
    LIMIT 5
""")

┌─────────┐
│ product │
│ varchar │
├─────────┤
│ Mouse   │
│ Mouse   │
│ Monitor │
│ Mouse   │
│ Monitor │
└─────────┘

In [15]:
# 4. NULL的特殊处理
# SQL里的NULL不是“等于任何东西”，包括它自己
# WHERE col = NULL ← 永远返回0行（错误写法）
# WHERE col IS NULL ← 正确写法
# WHERE col IS NOT NULL ← 非空

In [16]:
# 5. ORDER BY排序
# 升序（默认）
duckdb.sql("""
    SELECT order_id, total
    FROM '../data/sales.csv'
    ORDER BY total
    LIMIT 5
""")

┌──────────┬───────┐
│ order_id │ total │
│ varchar  │ int64 │
├──────────┼───────┤
│ O1026    │    99 │
│ O1115    │    99 │
│ O1119    │    99 │
│ O1045    │    99 │
│ O1083    │   198 │
└──────────┴───────┘

In [17]:
# 降序 DESC
duckdb.sql("""
    SELECT order_id, total
    FROM '../data/sales.csv'
    ORDER BY total DESC
    LIMIT 5
""")

┌──────────┬───────┐
│ order_id │ total │
│ varchar  │ int64 │
├──────────┼───────┤
│ O1169    │  9995 │
│ O1114    │  9995 │
│ O1104    │  9995 │
│ O1018    │  7996 │
│ O1066    │  7996 │
└──────────┴───────┘

In [18]:
# 多关键字排序（对应Python的sorted(key=lambda x: (a, -b))）
duckdb.sql("""
    SELECT customer_id, country, total
    FROM '../data/sales.csv'
    ORDER BY country ASC, total DESC
    LIMIT 10
""")

┌─────────────┬─────────┬───────┐
│ customer_id │ country │ total │
│   varchar   │ varchar │ int64 │
├─────────────┼─────────┼───────┤
│ C001        │ China   │  3996 │
│ C002        │ China   │  3897 │
│ C005        │ China   │  2598 │
│ C006        │ China   │  2396 │
│ C003        │ China   │  1999 │
│ C005        │ China   │  1196 │
│ C007        │ China   │   598 │
│ C001        │ China   │   495 │
│ C007        │ China   │   396 │
│ C004        │ China   │   299 │
└─────────────┴─────────┴───────┘
  10 rows             3 columns

In [19]:
# 6. DISTINCT去重
# 看数据集里有哪几个国家
duckdb.sql("""
    SELECT DISTINCT country
    FROM '../data/sales.csv'
""")

┌─────────┐
│ country │
│ varchar │
├─────────┤
│ Germany │
│ US      │
│ China   │
│ France  │
│ UK      │
└─────────┘

In [20]:
# DISTINCT多列：看“国家+品类”的所有组合
duckdb.sql("""
    SELECT DISTINCT country, category
    FROM '../data/sales.csv'
    ORDER BY country, category
""")

┌─────────┬───────────┐
│ country │ category  │
│ varchar │  varchar  │
├─────────┼───────────┤
│ China   │ Accessory │
│ China   │ Audio     │
│ China   │ Computer  │
│ France  │ Accessory │
│ France  │ Audio     │
│ France  │ Computer  │
│ France  │ Mobile    │
│ Germany │ Accessory │
│ Germany │ Audio     │
│ Germany │ Computer  │
│ Germany │ Mobile    │
│ UK      │ Accessory │
│ UK      │ Audio     │
│ UK      │ Computer  │
│ UK      │ Mobile    │
│ US      │ Accessory │
│ US      │ Audio     │
│ US      │ Computer  │
│ US      │ Mobile    │
└─────────┴───────────┘
  19 rows   2 columns

In [21]:
# 7. SQL的执行顺序
# SQL的实际执行顺序是：
#   1. FROM （确定数据源）
#   2. WHERE （先过滤掉不要的行）
#   3. SELECT （选要的列）
#   4. ORDER BY （排序）
#   5. LIMIT （取前N行）
# 这个顺序决定了：
# - WHERE里不能用SELECT起的别名（因为WHERE时还没SELECT）
# - ORDER BY里可以用SELECT起的别名（因为ORDER BY时已经SELECT了）